# Single Session Analysis

Analyze a single recording session:
- Event alignment and timing verification
- Lick analysis (tone-to-first-lick latency)
- Peri-tone lick PSTH (overall and chunked)
- Event raster plots
- Velocity and movement analysis
- Event counts over time

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt

from config import NODES, EDGES, NODE_IDX
from data_io import (
    load_aligned_data, load_sleap_dannce_keys, load_opcon_events,
    load_behavior_log, load_frame_mapping,
)
from skeleton import normalize_skeleton_batch
from processing import compute_speed
from visualization import plot_event_raster, plot_peri_event_psth

%matplotlib inline

In [ ]:
# --- Configuration ---
rat = 'R1'
session = '2025_10_28_1'

## Load session data

In [ ]:
aligned = load_aligned_data(rat, session)
keys = load_sleap_dannce_keys(rat, session)

sleap_3d = keys['sleap_keys_3D']
dannce_3d = keys['dannce_keys_3D']
if dannce_3d.ndim == 4:
    dannce_3d = dannce_3d.squeeze(axis=1).transpose(0, 2, 1)

tone_times = aligned['tone_times_ms'].ravel()
lick_times = aligned['lick_times_ms'].ravel()
reward_times = aligned['reward_times_ms'].ravel()
sleap_times = aligned['sleap_times_ms'].ravel()
dannce_times = aligned['dannce_times_ms'].ravel()

print(f'SLEAP frames: {len(sleap_times)}')
print(f'DANNCE frames: {len(dannce_times)}')
print(f'Tones: {len(tone_times)}, Licks: {len(lick_times)}, Rewards: {len(reward_times)}')

## Event raster

In [ ]:
fig, ax = plt.subplots(figsize=(18, 3))
plot_event_raster(ax, {
    'lick': lick_times,
    'tone': tone_times,
    'reward': reward_times,
})
ax.set_title(f'{rat}/{session} — Event raster')
plt.tight_layout()
plt.show()

## Lick analysis: tone-to-first-lick latency

In [ ]:
def compute_tone_to_lick_latency(tone_times_ms, lick_times_ms, max_latency_ms=10000):
    """For each tone, find the latency to the first subsequent lick."""
    latencies = []
    for t in tone_times_ms:
        post_licks = lick_times_ms[lick_times_ms > t]
        if len(post_licks) > 0:
            lat = post_licks[0] - t
            if lat <= max_latency_ms:
                latencies.append(lat)
            else:
                latencies.append(np.nan)
        else:
            latencies.append(np.nan)
    return np.array(latencies)


latencies = compute_tone_to_lick_latency(tone_times, lick_times)
valid = latencies[~np.isnan(latencies)]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(valid, bins=50, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Latency (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Tone-to-first-lick latency\nmean={np.nanmean(valid):.0f} ms, n={len(valid)}')

axes[1].plot(latencies, 'o', markersize=3, alpha=0.5)
axes[1].set_xlabel('Tone number')
axes[1].set_ylabel('Latency (ms)')
axes[1].set_title('Latency over trials')

plt.suptitle(f'{rat}/{session}')
plt.tight_layout()
plt.show()

## Peri-tone lick PSTH

In [ ]:
# Overall PSTH
fig, ax = plt.subplots(figsize=(8, 4))
plot_peri_event_psth(ax, tone_times, lick_times,
                     pre_ms=5000, post_ms=5000, bin_ms=500,
                     label='Licks')
ax.set_title(f'{rat}/{session} — Peri-tone lick PSTH')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Chunked PSTH (15-minute blocks)
chunk_ms = 15 * 60 * 1000
session_dur = max(tone_times[-1], lick_times[-1]) if len(tone_times) > 0 and len(lick_times) > 0 else 0
n_chunks = max(1, int(np.ceil(session_dur / chunk_ms)))

fig, axes = plt.subplots(1, n_chunks, figsize=(5 * n_chunks, 4), squeeze=False)

for ci in range(n_chunks):
    t_start = ci * chunk_ms
    t_end = (ci + 1) * chunk_ms
    chunk_tones = tone_times[(tone_times >= t_start) & (tone_times < t_end)]
    
    ax = axes[0, ci]
    plot_peri_event_psth(ax, chunk_tones, lick_times,
                         pre_ms=5000, post_ms=5000, bin_ms=500)
    ax.set_title(f'{ci * 15}-{(ci + 1) * 15} min')

plt.suptitle(f'{rat}/{session} — Chunked peri-tone lick PSTH')
plt.tight_layout()
plt.show()

## Event counts over time

In [ ]:
bin_ms = 5 * 60 * 1000  # 5-minute bins
all_times = np.concatenate([lick_times, reward_times]) if len(reward_times) > 0 else lick_times
edges_bins = np.arange(all_times.min(), all_times.max() + bin_ms, bin_ms)

lick_counts, _ = np.histogram(lick_times, bins=edges_bins)
reward_counts, _ = np.histogram(reward_times, bins=edges_bins)

bin_centers = (edges_bins[:-1] + edges_bins[1:]) / 2 / 1000 / 60  # minutes

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(bin_centers - 0.5, lick_counts, width=2, alpha=0.7, label='Licks', color='blue')
ax.bar(bin_centers + 0.5, reward_counts, width=2, alpha=0.7, label='Rewards', color='green')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Count per 5-min bin')
ax.set_title(f'{rat}/{session} — Event counts')
ax.legend()
plt.tight_layout()
plt.show()

## Velocity analysis

In [ ]:
# Compute speed of SpineM (back) keypoint
speed_sleap = compute_speed(sleap_3d, keypoint_idx=NODE_IDX['SpineM'])

# 1-minute binned average speed
fps_sleap = 1000.0 / np.median(np.diff(sleap_times)) if len(sleap_times) > 1 else 50
bin_frames = int(60 * fps_sleap)  # 1 minute

n_bins = len(speed_sleap) // bin_frames
binned_speed = [np.mean(speed_sleap[i * bin_frames:(i + 1) * bin_frames]) 
                for i in range(n_bins)]

fig, axes = plt.subplots(2, 1, figsize=(14, 6))

axes[0].plot(speed_sleap[:5000], alpha=0.5, linewidth=0.5)
axes[0].set_xlabel('Frame')
axes[0].set_ylabel('Speed (units/frame)')
axes[0].set_title(f'{rat}/{session} — SpineM speed (first 5000 frames)')

axes[1].plot(np.arange(n_bins), binned_speed, 'o-')
axes[1].set_xlabel('Time (min)')
axes[1].set_ylabel('Mean speed (units/frame)')
axes[1].set_title('Average speed per minute')

plt.tight_layout()
plt.show()

## Frame processing statistics

In [ ]:
try:
    frame_mapping = load_frame_mapping(rat, session)
    processed_frames = frame_mapping['processed_frame'].to_numpy()
    cam_frames = frame_mapping['cam0_frame'].to_numpy()
    
    n_total_sleap = len(sleap_times)
    n_processed = len(processed_frames)
    skip_rate = 1 - (n_processed / n_total_sleap) if n_total_sleap > 0 else 0
    
    print(f'Total SLEAP camera frames: {n_total_sleap}')
    print(f'Processed keypoint frames: {n_processed}')
    print(f'Skip rate: {skip_rate:.1%}')
    
    # Show frame gaps
    if len(cam_frames) > 1:
        gaps = np.diff(cam_frames)
        fig, ax = plt.subplots(figsize=(10, 3))
        ax.plot(gaps, linewidth=0.5)
        ax.set_xlabel('Processed frame index')
        ax.set_ylabel('Camera frame gap')
        ax.set_title(f'Frame gaps (mean={np.mean(gaps):.1f}, max={np.max(gaps)})')
        plt.tight_layout()
        plt.show()
except Exception as e:
    print(f'Could not load frame mapping: {e}')